# Week 1 - Data collection, loading and initial inspection

Active scope: the official RouteRanker brief, Week 1. New York is the user-approved alternative to its original Indian-city scope. This notebook does not clean source records, engineer features, run EDA, train models or build an application.

Run `python scripts/week1_inspection.py` first to generate full-table checks. This notebook validates the same frozen inputs, then explains their saved inspection evidence.

In [1]:
from pathlib import Path
import sys,json,zipfile
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"scripts/week1_inspection.py").exists())
sys.path.insert(0,str(ROOT/"scripts"))
import week1_inspection as week1
print("Project:",ROOT)
print("Active stage: Week 1 only")

Project: R:\education\router
Active stage: Week 1 only


## 1. Verify collected source files

Each receipt records its URL/query, retrieval time, bytes and SHA-256. A matching fingerprint establishes file identity, not correctness of the publisher. Downloaded hourly/daily aggregates are exact publisher query responses, not ticket-level records.

In [2]:
summary=json.loads((week1.REPORT/"inspection_summary.json").read_text())
expected={row["local_path"]:row["sha256"] for row in pd.read_csv(week1.REPORT/"source_inventory.csv").to_dict("records")}
receipts=week1.inventory()
actual={row["local_path"]:row["sha256"] for row in receipts}
assert actual==expected,"Inputs changed: rerun the full Week 1 inspection before using these reports."
print("Verified source receipts:",len(receipts))
display(pd.DataFrame(receipts)[["local_path","bytes","sha256_verified"]].head(8))

Verified source receipts: 57


,local_path,bytes,sha256_verified
0,data/raw/new_york/gtfs/archive/manhattan_20251...,9647917,True
1,data/raw/new_york/gtfs/archive/manhattan_20260...,9593304,True
2,data/raw/new_york/gtfs/archive/manhattan_20260...,7762091,True
3,data/raw/new_york/gtfs/archive/manhattan_20260...,6708453,True
4,data/raw/new_york/gtfs/archive/manhattan_20260...,9519944,True
5,data/raw/new_york/gtfs/current/bronx.zip,8428753,True
6,data/raw/new_york/gtfs/current/brooklyn.zip,17046285,True
7,data/raw/new_york/gtfs/current/bus_company.zip,8446523,True


## 2. Load the essential GTFS files

Read the original ZIP directly into pandas; extraction to a separate text folder is unnecessary. IDs are deliberately kept as strings. The five previews below are bounded; the saved inspection reports cover all rows, not just these previews.

In [3]:
source=week1.RAW/"gtfs/current/manhattan.zip"
with zipfile.ZipFile(source) as archive:
    routes=pd.read_csv(archive.open("routes.txt"),dtype="string")
    trips_preview=pd.read_csv(archive.open("trips.txt"),dtype="string",nrows=5)
    stop_times_preview=pd.read_csv(archive.open("stop_times.txt"),dtype="string",nrows=5)
    stops_preview=pd.read_csv(archive.open("stops.txt"),dtype="string",nrows=5)
    calendar=pd.read_csv(archive.open("calendar.txt"),dtype="string")
print("Manhattan catalogue routes:",routes.shape)
routes.info(memory_usage="deep")
for name,frame in [("routes",routes.head()),("trips",trips_preview),("stop_times",stop_times_preview),("stops",stops_preview),("calendar",calendar.head())]:
    print(name)
    display(frame)

Manhattan catalogue routes: (307, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307 entries, 0 to 306
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   route_id          307 non-null    string
 1   agency_id         307 non-null    string
 2   route_short_name  307 non-null    string
 3   route_long_name   307 non-null    string
 4   route_desc        300 non-null    string
 5   route_type        307 non-null    string
 6   route_color       307 non-null    string
 7   route_text_color  307 non-null    string
dtypes: string(8)
memory usage: 163.2 KB
routes


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color
0,B1,MTA NYCT,B1,Bay Ridge - Manhattan Beach,via 86th St / Ocean Pkwy,3,00AEEF,FFFFFF
1,B11,MTA NYCT,B11,Sunset Park - Midwood,via 49th & 50th St / Avenue J,3,006CB7,FFFFFF
2,B12,MTA NYCT,B12,Lefferts Gardens - East New York,via Clarkson Av / Empire Blvd / East New York Av,3,6CBE45,FFFFFF
3,B13,MTA NYCT,B13,Spring Creek - Wyckoff Hospital,via Crescent St / Jamaica Av / Wyckoff Av,3,FAA61A,FFFFFF
4,B14,MTA NYCT,B14,Spring Creek - Crown Heights,via Sutter Av / Pitkin Av,3,00AEEF,FFFFFF


trips


,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id
0,M60+,MQ_O6-Weekday,MQ_O6-Weekday-009000_SBS60_901,SELECT BUS LA GUARDIA AIRPORT,0,39855329,SBS600010
1,M60+,MQ_O6-Weekday,MQ_O6-Weekday-013000_SBS60_901,SELECT BUS WEST SIDE BROADWAY-106 ST,1,39855329,SBS600011
2,M60+,MQ_O6-Weekday,MQ_O6-Weekday-019000_SBS60_901,SELECT BUS LA GUARDIA AIRPORT,0,39855329,SBS600010
3,M60+,MQ_O6-Weekday,MQ_O6-Weekday-022800_SBS60_901,SELECT BUS WEST SIDE BROADWAY-106 ST,1,39855329,SBS600011
4,M60+,MQ_O6-Weekday,MQ_O6-Weekday-029500_SBS60_906,SELECT BUS LA GUARDIA AIRPORT,0,39855337,SBS600010


stop_times


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,timepoint
0,MQ_O6-Weekday-009000_SBS60_901,01:30:00,01:30:00,403117,1,0,0,1
1,MQ_O6-Weekday-009000_SBS60_901,01:33:11,01:33:11,403122,2,0,0,0
2,MQ_O6-Weekday-009000_SBS60_901,01:34:31,01:34:31,404262,3,0,0,0
3,MQ_O6-Weekday-009000_SBS60_901,01:35:51,01:35:51,401442,4,0,0,0
4,MQ_O6-Weekday-009000_SBS60_901,01:37:32,01:37:32,405424,5,0,0,0


stops


,stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station
0,101014,WILLIS AV/E 138 ST,<NA>,40.808930,-73.922855,<NA>,<NA>,0,<NA>
1,101015,WILLIS AV/E 140 ST,<NA>,40.810634,-73.921632,<NA>,<NA>,0,<NA>
2,101017,WILLIS AV/E 144 ST,<NA>,40.813111,-73.919823,<NA>,<NA>,0,<NA>
3,101018,WILLIS AV/E 146 ST,<NA>,40.814404,-73.918954,<NA>,<NA>,0,<NA>
4,101066,3 AV/E 150 ST,<NA>,40.816142,-73.917547,<NA>,<NA>,0,<NA>


calendar


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,MQ_D6-Weekday-SDon,1,1,1,1,1,0,0,20261009,20261223
1,MQ_D6-Saturday,0,0,0,0,0,1,0,20261010,20270102
2,MQ_D6-Sunday,0,0,0,0,0,0,1,20261011,20270101
3,MQ_O6-Weekday,1,1,1,1,1,0,0,20261012,20261231
4,MV_D6-Weekday-SDon,1,1,1,1,1,0,0,20261009,20261223


## 3. Dataset coverage and dimensions

Different borough feeds can share route catalogues. Catalogue rows do not equal active routes. Trip rows are schedule templates, not passengers or automatically daily trip counts. Calendar dates define planned applicability, not observed operations.

In [4]:
inventory=pd.read_csv(week1.REPORT/"gtfs_inventory.csv")
display(inventory[["feed","routes","trips","stop_times","stops","calendar","routes_referenced_by_trips","calendar_start","calendar_end"]])
display(pd.DataFrame([summary["hourly_source"],summary["daily_source"]])[["resolution","rows","columns","dates","published_route_identifiers","first_date","last_date"]])

,feed,routes,trips,stop_times,stops,calendar,routes_referenced_by_trips,calendar_start,calendar_end
0,archive_manhattan_20251230,294,37864,1177546,1828,21,41,2026-01-03,2026-03-28
1,archive_manhattan_20260327,294,37864,1177546,1828,21,41,2026-01-03,2026-04-11
2,archive_manhattan_20260409,294,30289,947151,1830,16,41,2026-04-11,2026-06-27
3,archive_manhattan_20260624,307,30285,946079,1825,16,41,2026-06-27,2026-09-05
4,archive_manhattan_20260903,307,37872,1174914,1824,21,41,2026-09-05,2027-01-02
5,current_bronx,307,41156,1160917,1890,15,46,2026-10-08,2027-01-02
6,current_brooklyn,307,58357,2399470,4575,36,68,2026-10-08,2027-01-02
7,current_bus_company,92,45424,1077998,2758,32,92,2026-10-09,2027-01-02
8,current_manhattan,307,37872,1174640,1823,21,41,2026-10-08,2027-01-02
9,current_queens,307,31199,817866,1422,20,40,2026-10-08,2027-01-02


,resolution,rows,columns,dates,published_route_identifiers,first_date,last_date
0,route/local clock hour,993600,5,92,450,2026-07-01,2026-09-30
1,route/local clock date,275896,6,638,450,2025-01-01,2026-09-30


## 4. Initial quality checks

Inspect missingness, loaded types, memory, duplicates, keys, references and basic formats. Missing optional fields are reported rather than removed. Coordinate ranges do not prove road/route correctness. GTFS times may exceed 24:00:00.

In [5]:
folder=week1.REPORT/"gtfs/current_manhattan"
display(pd.read_csv(folder/"table_inspection.csv"))
display(pd.read_csv(folder/"column_inspection.csv"))
display(pd.read_csv(folder/"key_and_field_checks.csv"))

,table,rows,columns,duplicate_full_rows,missing_cells,memory_bytes
0,agency,1,6,0,0,557
1,calendar,21,10,0,0,12899
2,calendar_dates,310,3,0,0,60258
3,routes,307,8,0,7,167128
4,shapes,33865,4,0,0,8809442
5,stops,1823,9,0,7292,1072932
6,stop_times,1174640,8,0,0,601263282
7,trips,37872,7,0,0,18707878
8,feed_info,1,7,0,0,664


,table,column,loaded_dtype,rows,missing_count,missing_percent,distinct_nonmissing,memory_bytes
0,agency,agency_id,string,1,0,0.0,1,65
1,agency,agency_name,string,1,0,0.0,1,82
2,agency,agency_url,string,1,0,0.0,1,77
3,agency,agency_timezone,string,1,0,0.0,1,73
4,agency,agency_lang,string,1,0,0.0,1,59
...,...,...,...,...,...,...,...,...
57,feed_info,feed_lang,string,1,0,0.0,1,59
58,feed_info,feed_start_date,string,1,0,0.0,1,65
59,feed_info,feed_end_date,string,1,0,0.0,1,65
60,feed_info,feed_version,string,1,0,0.0,1,65


,check,count,scope
0,routes_duplicate_keys,0,"Duplicate key rows, not duplicate display names"
1,trips_duplicate_keys,0,"Duplicate key rows, not duplicate display names"
2,stops_duplicate_keys,0,"Duplicate key rows, not duplicate display names"
3,stop_times_duplicate_keys,0,"Duplicate key rows, not duplicate display names"
4,calendar_duplicate_keys,0,"Duplicate key rows, not duplicate display names"
5,duplicate_service_date_exceptions,0,Calendar exception keys
6,invalid_exception_dates,0,Calendar exception date parse
7,invalid_exception_types,0,Expected addition/removal flags
8,unknown_trip_routes,0,Trip route foreign keys
9,unknown_trip_services,0,Calendar plus exception service IDs


## 5. Ridership source inspection and limitations

The hourly pilot covers three months; the additional collected history is daily. Electronic-fare estimates may contain publisher imputation and exclude cash/free travel. Transfers are already included. Separate publisher daily queries verify extraction arithmetic, not true passenger totals. Original fare-category checks cover only a bounded sample.

In [6]:
display(pd.read_csv(week1.REPORT/"hourly_source_column_inspection.csv"))
display(pd.read_csv(week1.REPORT/"daily_source_column_inspection.csv"))
print("Bounded raw sample:")
print(json.dumps(summary["bounded_raw_sample"],indent=2))
print(summary["hourly_source"]["reconciliation"])
print(summary["daily_source"]["reconciliation"])

,table,column,loaded_dtype,rows,missing_count,missing_percent,distinct_nonmissing,memory_bytes
0,hourly_source,transit_timestamp,object,993600,0,0.0,2208,79488000
1,hourly_source,bus_route,string,993600,0,0.0,450,60044352
2,hourly_source,ridership,int64,993600,0,0.0,1899,7948800
3,hourly_source,transfers,int64,993600,0,0.0,678,7948800
4,hourly_source,source_rows,int64,993600,0,0.0,1,7948800


,table,column,loaded_dtype,rows,missing_count,missing_percent,distinct_nonmissing,memory_bytes
0,daily_source,service_date,object,275896,0,0.0,638,22071680
1,daily_source,bus_route,string,275896,0,0.0,450,16666328
2,daily_source,published_boardings,int64,275896,0,0.0,15465,2207168
3,daily_source,published_transfers,int64,275896,0,0.0,4987,2207168
4,daily_source,reported_hours,int64,275896,0,0.0,1,2207168
5,daily_source,source_rows,int64,275896,0,0.0,2,2207168


Bounded raw sample:
{
  "rows": 10000,
  "duplicate_full_rows": 0,
  "missing_values": {
    "transit_timestamp": 0,
    "bus_route": 0,
    "payment_method": 0,
    "fare_class_category": 0,
    "ridership": 0,
    "transfers": 0
  },
  "scope": "Bounded 10,000-row sample only, not an audit of every original fare-category row."
}
Passed against separate daily query of the same publisher; not independent passenger measurement
Passed against separate daily query of the same publisher; not independent passenger measurement


## Week 1 handoff

Collection, essential-file loading and initial inspection are complete for the approved New York scope. The original Lepton/23-Indian-city scope is not fulfilled. No cleaning, route-frequency/duration features, EDA, scaling, models, scenarios, database/API or dashboard progress is claimed here.

Read `reports/week1/WEEK_1_REPORT.md`. Week 2 remains a separate next step; Streamlit is the fixed Week 7 dashboard, with FastAPI/PostgreSQL allowed later.